# 动作执行系统 (Action Execution System)

## 学习目标

本教程将帮助你理解自主智能体中的动作执行系统：

1. **Action 数据结构** - 动作的表示与类型
2. **ActionHandler** - 不同类型动作的处理器
3. **ActionExecutor** - 统一的执行接口
4. **重试与错误处理** - 指数退避策略

---

In [ ]:
# 环境设置
import sys
sys.path.insert(0, '../src')

from action_executor import (
    Action, ActionType, ActionStatus, ActionResult,
    ActionExecutor, ToolAction, CodeAction, ThinkAction,
    create_action
)

## 1. Action 数据结构

### 1.1 动作类型

In [ ]:
# 查看所有动作类型
print("支持的动作类型:")
for action_type in ActionType:
    print(f"  - {action_type.value}")

In [ ]:
# 创建动作
action = Action(
    action_type=ActionType.TOOL,
    name="calculator",
    parameters={"operation": "add", "a": 5, "b": 3}
)

print(f"动作 ID: {action.action_id}")
print(f"类型: {action.action_type.value}")
print(f"名称: {action.name}")
print(f"参数: {action.parameters}")

In [ ]:
# 使用工厂函数创建动作
action = create_action("code", "execute", code="result = 2 ** 10")
print(f"类型: {action.action_type.value}")
print(f"参数: {action.parameters}")

## 2. 动作处理器

### 2.1 ToolAction - 工具调用

In [ ]:
# 创建工具处理器
tool_handler = ToolAction()

# 注册工具
tool_handler.register_tool("add", lambda a, b: a + b)
tool_handler.register_tool("multiply", lambda x, y: x * y)
tool_handler.register_tool("greet", lambda name: f"Hello, {name}!")

print("已注册的工具:", tool_handler.list_tools())

In [ ]:
# 执行工具
action = Action(ActionType.TOOL, "add", {"a": 10, "b": 20})
result = tool_handler.execute(action)

print(f"执行: add(10, 20)")
print(f"状态: {result.status.value}")
print(f"结果: {result.output}")

In [ ]:
# 工具不存在的情况
action = Action(ActionType.TOOL, "unknown_tool", {})
result = tool_handler.execute(action)

print(f"状态: {result.status.value}")
print(f"错误: {result.error}")

### 2.2 CodeAction - 代码执行

在沙箱环境中安全执行 Python 代码：

In [ ]:
code_handler = CodeAction()

# 执行简单计算
action = Action(ActionType.CODE, "exec", {"code": "result = sum(range(1, 101))"})
result = code_handler.execute(action)

print("代码: result = sum(range(1, 101))")
print(f"结果: {result.output}")

In [ ]:
# 执行更复杂的代码
code = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

result = [fibonacci(i) for i in range(10)]
"""

action = Action(ActionType.CODE, "exec", {"code": code})
result = code_handler.execute(action)

print("斐波那契数列前10项:")
print(f"结果: {result.output}")

In [ ]:
# 语法错误处理
action = Action(ActionType.CODE, "exec", {"code": "def broken("})
result = code_handler.execute(action)

print(f"状态: {result.status.value}")
print(f"错误: {result.error}")

### 2.3 ThinkAction - 思考记录

记录 Agent 的思考过程（无副作用）：

In [ ]:
think_handler = ThinkAction()

action = Action(ActionType.THINK, "analyze", {
    "thought": "用户需要一个排序算法，考虑到数据量较小，使用快速排序即可"
})
result = think_handler.execute(action)

print(f"思考: {result.output}")
print(f"状态: {result.status.value}")

## 3. ActionExecutor 统一接口

### 3.1 基本使用

In [ ]:
# 创建执行器
executor = ActionExecutor()

# 注册自定义工具
for handler in executor.registry._handlers:
    if isinstance(handler, ToolAction):
        handler.register_tool("square", lambda x: x ** 2)
        handler.register_tool("reverse", lambda s: s[::-1])
        break

In [ ]:
# 执行不同类型的动作
actions = [
    Action(ActionType.TOOL, "square", {"x": 7}),
    Action(ActionType.CODE, "exec", {"code": "result = 'hello'.upper()"}),
    Action(ActionType.THINK, "plan", {"thought": "下一步应该验证结果"}),
]

for action in actions:
    result = executor.execute(action)
    print(f"[{action.action_type.value}] {action.name}: {result.output}")

### 3.2 执行统计

In [ ]:
# 查看执行统计
stats = executor.get_stats()
print("执行统计:")
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# 查看执行历史
history = executor.get_history()
print(f"\n执行历史 ({len(history)} 条):")
for record in history:
    print(f"  {record['action_type']}: {record['status']}")

## 4. 重试与错误处理

### 4.1 指数退避算法

$$\text{delay}(n) = \min(\text{base} \times 2^n + \text{jitter}, \text{max\_delay})$$

In [ ]:
import random

def calculate_backoff(attempt, base_delay=1.0, max_delay=60.0):
    """计算指数退避延迟"""
    delay = base_delay * (2 ** attempt)
    jitter = random.uniform(0, 0.1 * delay)
    return min(delay + jitter, max_delay)

print("指数退避延迟示例:")
for attempt in range(6):
    delay = calculate_backoff(attempt)
    print(f"  尝试 {attempt}: {delay:.2f} 秒")

### 4.2 ActionResult 结构

In [ ]:
# 成功结果
success_result = ActionResult(
    status=ActionStatus.SUCCESS,
    output="计算完成",
    execution_time=0.05
)
print(f"成功: is_success={success_result.is_success}")

# 失败结果
failed_result = ActionResult(
    status=ActionStatus.FAILED,
    error="连接超时",
    execution_time=30.0
)
print(f"失败: is_success={failed_result.is_success}, error={failed_result.error}")

## 5. 实践示例

### 5.1 构建计算器工具集

In [ ]:
import math

# 创建新的执行器
calc_executor = ActionExecutor()

# 获取 ToolAction 处理器并注册计算器工具
for handler in calc_executor.registry._handlers:
    if isinstance(handler, ToolAction):
        handler.register_tool("add", lambda a, b: a + b)
        handler.register_tool("subtract", lambda a, b: a - b)
        handler.register_tool("multiply", lambda a, b: a * b)
        handler.register_tool("divide", lambda a, b: a / b if b != 0 else "Error: Division by zero")
        handler.register_tool("sqrt", lambda x: math.sqrt(x))
        handler.register_tool("power", lambda base, exp: base ** exp)
        break

# 执行一系列计算
calculations = [
    ("add", {"a": 10, "b": 5}),
    ("multiply", {"a": 3, "b": 7}),
    ("sqrt", {"x": 144}),
    ("power", {"base": 2, "exp": 8}),
]

print("计算器演示:")
for name, params in calculations:
    action = Action(ActionType.TOOL, name, params)
    result = calc_executor.execute(action)
    print(f"  {name}{tuple(params.values())} = {result.output}")

## 6. 练习

### 练习 1: 创建字符串处理工具集

In [ ]:
# 你的代码: 注册 upper, lower, reverse, count_words 等工具
# ...

### 练习 2: 实现带重试的执行

In [ ]:
# 你的代码: 实现一个会随机失败的工具，并使用重试机制
# ...

## 总结

| 组件 | 功能 | 关键方法 |
|:-----|:-----|:---------|
| `Action` | 动作数据结构 | `action_type`, `parameters` |
| `ToolAction` | 工具调用 | `register_tool()`, `execute()` |
| `CodeAction` | 代码执行 | `execute()` (沙箱环境) |
| `ThinkAction` | 思考记录 | `execute()` (无副作用) |
| `ActionExecutor` | 统一执行 | `execute()`, `get_stats()` |

下一教程将介绍完整的自主智能体系统。